# Time-Frequency EMRI Waveforms with fewtrax
---
T = 2 yr, WDM grid: Nt \simeq 1024 (~1-day bins), Nf = 2048 (df $\sim 8 \mu$Hz)


## How the WDM waveform is constructed

### Standard time-domain route
For an EMRI waveform with $N_\text{modes}$ harmonic modes sampled at $\Delta t \sim 10$ s over $T = 2$ yr, the strain is:
$$h(t_i) = \sum_{\ell m k n} \left[\,{}_{-2}Y_{\ell m}(\theta,\phi)\;A_{\ell m k n}(t_i)\;e^{-i\Phi_{mkn}(t_i)} + \text{c.c.}\right]$$
at $N_\text{dense} = T/\Delta t \sim 6\times 10^6$ time steps — costing $O(N_\text{dense}\times N_\text{modes})$ operations.

### WDM basis
The Wilson–Daubechies–Meyer (WDM) basis provides an orthonormal tiling of the time-frequency plane.  Each basis function $\psi_{qn}(t)$ is localised at time $t_n = n\,\Delta T$ and frequency $f_q = q\,\Delta F$ (meaning it is, by construction, uniform in frequencies and not in log-frequency. Thus we have lower resolution for low-frequency sources, though not really a problem for EMRIs) with:
$$\Delta T = T/N_t, \qquad \Delta F = 1/(2\Delta T), \qquad N = N_f N_t$$
The WDM coefficient of $h$ is:
$$W[q,n] = \int h(t)\,\psi_{qn}^*(t)\,dt$$

### Direct WDM formula
For a slowly-chirping harmonic with instantaneous frequency $f_{mkn}(t)$, amplitude $A_{mkn}(t)$, and phase $\Phi_{mkn}(t)$, the stationary-phase approximation gives:
$$W[q,n] \approx A_{mkn}(t_n)\;e^{-i\Phi_{mkn}(t_n)}\cdot K\!\left(\frac{q\Delta F - f_{mkn}(t_n)}{\Delta F},\;\dot{f}_{mkn}\Delta T^2\right)$$
where the complex Meyer kernel with leading-order Fresnel chirp correction is:
$$K(\xi,\chi) = K_0(\xi)\;e^{-i\pi\chi\xi^2}, \qquad K_0(\xi) = \begin{cases} 1 & |\xi|\le\tfrac{1}{2} \\ \cos\!\left(\tfrac{\pi}{2}\nu(2|\xi|-1)\right) & \tfrac{1}{2}<|\xi|\le 1 \\ 0 & |\xi|>1 \end{cases}$$

**Key saving**: only $N_t$ evaluations per mode (at WDM bin centres) instead of $N_\text{dense} = N_f\times N_t$ — a factor of $N_f \sim 10^3$ reduction.

### Sparsity of EMRI tracks
Each harmonic $f_{mkn}(t)$ evolves slowly (adiabatically), so each mode sweeps through $\sim N_t$ pixels arranged along a narrow track: a fraction $1/N_f$ of all $N_f N_t$ pixels.  The full signal is concentrated in $\sim N_\text{modes}\times N_t$ active pixels.

### Implementation in fewtrax
1. The ODE integrator `EMRIInspiral` provides $p(t), e(t), \Phi(t)$ (~200 sparse points).
2. Cubic-spline interpolation maps all quantities to the $N_t$ WDM bin centres.
3. Instantaneous frequencies are computed from `get_fundamental_frequencies` evaluated on the interpolated $(p,e)$ using an analytical formula.
4. `jax.vmap` batches `direct_wdm_mode` over all $N_\text{modes}$ simultaneously and so a single scatter-add fills the $N_f\times N_t$ grid.

The full operation is covered by `direct_wdm_sum` in `fewtrax.summation.tf_sum`.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join("..", ".env"))
except ImportError:
    pass

import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LogNorm, Normalize

from fewtrax import KerrEccentricEquatorialWaveform
from fewtrax.data.loader import load_flux_data
from fewtrax.trajectory.inspiral import EMRIInspiral
from fewtrax.utils.tf_tracks import (
    WDMGrid, TFTrack, TFTrackSet,
    analytical_tf_track, compute_freq_track, compute_freq_track_batch,
    build_tf_tracks, sparse_wdm_track, direct_wdm_mode,
)
from fewtrax.summation.tf_sum import direct_wdm_sum
from fewtrax.utils.harmonics import get_ylms_for_modes
from fewtrax.utils.constants import MTSUN_SI, YEAR_SI, GPC_SI, G_SI, C_SI, MSUN_SI
from fewtrax.waveform.kerr import _get_viewing_angles

try:
    from pywavelet.transforms.numpy.forward.main import from_time_to_wavelet
    from pywavelet.types import TimeSeries
    HAS_PYWAVELET = True
    print("pywavelet: OK")
except ImportError:
    HAS_PYWAVELET = False
    print("pywavelet not found: FEW comparison and WDM track cells will be skipped")

try:
    from few.waveform import GenerateEMRIWaveform
    HAS_FEW = True
    print("FEW: OK")
except ImportError:
    HAS_FEW = False
    print("FEW not found: comparison will use fewtrax time-domain as reference")

print(f"JAX backend: {jax.default_backend()}, devices: {jax.devices()}")

ImportError: cannot import name 'direct_wdm_mode' from 'fewtrax.utils.tf_tracks' (/Users/bertd/Documents/PhD/LISA/Projects/JAX-waveform/fewtrax/src/fewtrax/utils/tf_tracks.py)

In [4]:
#Physical parameters
M      = 1e6    # primary mass [M_sun]
mu     = 10.0   # secondary mass [M_sun]
a      = 0.3    # Kerr spin
p0     = 10.0   # initial semi-latus rectum [M]
e0     = 0.4    # initial eccentricity
x0     = 1.0    # prograde
dist   = 1.0    # luminosity distance [Gpc]
qS     = 0.2;  phiS = 0.2    # sky angles [rad]
qK     = 0.8;  phiK = 0.8    # spin orientation [rad]
Phi_phi0   = 1.0
Phi_theta0 = 2.0
Phi_r0     = 3.0
T_yr   = 2.0    # requested observation time [years]

In [2]:
# Data directory 
data_dir = os.environ.get("FEW_DATA_DIR")
if data_dir is None or not os.path.isdir(str(data_dir)):
    raise FileNotFoundError(
        "Set FEW_DATA_DIR in .env or shell environment. "
        "See comparison/utils.py for the lookup chain."
    )
print(f"Data directory: {data_dir}")

Data directory: /Users/bertd/Documents/PhD/LISA/Codes/FastEMRIWaveforms/src/few/data


In [5]:
# WDM grid (nominal, will be updated to T_actual after trajectory)
# Nt = 1024  ->  dT = T/Nt ≈ 61 600 s ≈ 0.71 days  (sparse time bins)
# Nf = 2048  ->  dF = 1/(2dT) ≈ 8.1 μHz             (high frequency resolution)
#              f_nyq = Nf·dF ≈ 16.6 mHz             (covers EMRI band)
Nt = 1024
Nf = 2048

T_s_nominal = T_yr * YEAR_SI
grid_nominal = WDMGrid(Nf=Nf, Nt=Nt, T=T_s_nominal)
print(f"\nNominal WDM grid: {grid_nominal}")
print(f"  Time bin width:  dT = {grid_nominal.delta_T/86400:.3f} days")
print(f"  Freq resolution: dF = {grid_nominal.delta_F*1e6:.2f} \u03bcHz")
print(f"  Nyquist freq:    f_nyq = {grid_nominal.f_nyq*1e3:.2f} mHz")
print(f"  Equiv. dt_wdm:   {grid_nominal.T/(Nf*Nt):.1f} s")


Nominal WDM grid: WDMGrid(Nf=2048, Nt=1024, T=6.312e+07s, dT=61637.0s, dF=8.11μHz, f_nyq=16.61mHz)
  Time bin width:  dT = 0.713 days
  Freq resolution: dF = 8.11 μHz
  Nyquist freq:    f_nyq = 16.61 mHz
  Equiv. dt_wdm:   30.1 s


In [ ]:
# Load waveform generator + compute sparse trajectory
print("Loading KerrEccentricEquatorialWaveform …")
t0 = time.perf_counter()
wf = KerrEccentricEquatorialWaveform(
    data_dir=data_dir,
    dense_steps=200,
    mode_selection_threshold=1e-5,
)
t_load = time.perf_counter() - t0
print(f"  Loaded in {t_load:.2f} s")

Loading KerrEccentricEquatorialWaveform …


u_A knot grid is not uniform (max deviation 3.125e-02); snapping to linspace over [0, 1] with 31 points.
w_A knot grid is not uniform (max deviation 3.125e-02); snapping to linspace over [0, 1] with 31 points.


In [ ]:
print("\nGenerating sparse trajectory …")
t0 = time.perf_counter()
sparse = wf.generate_sparse(
    M=M, mu=mu, a=a, p0=p0, e0=e0, x0=x0,
    T=T_yr, dt=10.0,
    Phi_phi0=Phi_phi0, Phi_theta0=Phi_theta0, Phi_r0=Phi_r0,
)
t_sparse = time.perf_counter() - t0

t_traj    = np.asarray(sparse["t"])
p_traj    = np.asarray(sparse["p"])
e_traj    = np.asarray(sparse["e"])
Phi_phi   = np.asarray(sparse["Phi_phi"])
Phi_theta = np.asarray(sparse["Phi_theta"])
Phi_r     = np.asarray(sparse["Phi_r"])
teuk_modes = np.asarray(sparse["teuk_modes"])
l_arr = sparse["l_arr"]
m_arr = sparse["m_arr"]
k_arr = sparse["k_arr"]
n_arr = sparse["n_arr"]
N_modes = len(m_arr)

valid    = np.isfinite(t_traj)
T_actual = float(t_traj[valid][-1])

print(f"  Trajectory: {valid.sum()} points, T_actual = {T_actual/YEAR_SI:.3f} yr")
print(f"  Mode count: {N_modes}")
print(f"  Wall time: {t_sparse:.2f} s")

In [ ]:
# Update grid to actual inspiral duration
grid = WDMGrid(Nf=Nf, Nt=Nt, T=T_actual)
dt_wdm = T_actual / (Nf * Nt)
print(f"\nActual WDM grid: {grid}")
print(f"  Effective dt_wdm = {dt_wdm:.1f} s  (time-domain equiv.)")

# Viewing angles and spherical harmonics
theta_obs, phi_obs = _get_viewing_angles(qS, phiS, qK, phiK)
ylms_pos, ylms_neg = get_ylms_for_modes(l_arr, m_arr, theta_obs, phi_obs)
print(f"\nViewing angle: \u03b8_obs = {np.degrees(theta_obs):.1f}\u00b0")

## 1. Single TF Track on a Sparse Time Grid

Each harmonic mode $(\ell, m, k, n)$ sweeps a slowly-chirping track through the WDM pixel plane.  
`analytical_tf_track` computes this track by interpolating $\Omega_\phi(t), \Omega_r(t)$ from the sparse trajectory — no WDM transform needed.  
`sparse_wdm_track` additionally generates the single-mode time series, applies the pywavelet WDM transform, and stores the pixel amplitudes along the track.

In [ ]:
DOMINANT_MODE = (2, 2, 0, 1)

# Analytical track (frequency only, no transform)
t0 = time.perf_counter()
track_anl = analytical_tf_track(
    DOMINANT_MODE, t_traj, p_traj, e_traj,
    a, M, mu, grid, x0=x0,
)
t_anl = time.perf_counter() - t0
print(f"Analytical track {DOMINANT_MODE}:  {t_anl*1e3:.1f} ms")
print(f"  f range: {track_anl.freq_hz.min()*1e3:.3f} – {track_anl.freq_hz.max()*1e3:.3f} mHz")
print(f"  Bin range: {int(track_anl.i_freq.min())} – {int(track_anl.i_freq.max())}")
print(f"  Memory: {track_anl.nbytes/1024:.1f} kB")

# Full WDM track via pywavelet (exact WDM coefficients)
track_wdm = None
t_wdm_track = None

mode_idx = None
for i, (l, m, k, n) in enumerate(zip(l_arr, m_arr, k_arr, n_arr)):
    if (int(l), int(m), int(k), int(n)) == DOMINANT_MODE:
        mode_idx = i
        break

if mode_idx is not None and HAS_PYWAVELET:
    print(f"\nComputing WDM track via pywavelet (mode_idx={mode_idx}) …")
    t0 = time.perf_counter()
    track_wdm = sparse_wdm_track(
        DOMINANT_MODE,
        t_traj, p_traj, e_traj,
        Phi_phi, Phi_theta, Phi_r,
        teuk_modes[:, mode_idx],
        a, M, mu, grid, x0=x0, hw=2,
    )
    t_wdm_track = time.perf_counter() - t0
    print(f"  WDM track time: {t_wdm_track:.2f} s")
    print(f"  {track_wdm}")
elif not HAS_PYWAVELET:
    print("\n(pywavelet unavailable — WDM track skipped)")
else:
    print(f"\n(Mode {DOMINANT_MODE} not in selected set — WDM track skipped)")

In [ ]:
t_bins_yr = grid.t_bins / YEAR_SI
dF_mhz    = grid.delta_F * 1e3
f_track   = track_anl.freq_hz * 1e3   # mHz

n_cols = 2 if track_wdm is not None else 1
fig, axes = plt.subplots(1, n_cols, figsize=(7*n_cols, 5))
ax0 = axes[0] if n_cols > 1 else axes

ax0.plot(t_bins_yr, f_track, color="C0", lw=1.5, label=f"Mode {DOMINANT_MODE}")
f_lo_ax = f_track.min() - 4*dF_mhz
f_hi_ax = f_track.max() + 4*dF_mhz
for q in range(Nf):
    f_line = q * dF_mhz
    if f_lo_ax <= f_line <= f_hi_ax:
        ax0.axhline(f_line, color="lightgray", lw=0.4, zorder=0)
ax0.set_xlim(0, T_actual/YEAR_SI)
ax0.set_ylim(f_lo_ax, f_hi_ax)
ax0.set_xlabel("Time [yr]")
ax0.set_ylabel("GW frequency [mHz]")
ax0.set_title(f"Analytical TF Track  {DOMINANT_MODE}")
ax0.legend()
ax0.text(
    0.02, 0.97,
    f"dT = {grid.delta_T/86400:.2f} d\ndF = {dF_mhz*1e3:.1f} \u03bcHz\nNt={Nt}, Nf={Nf}",
    transform=ax0.transAxes, va="top", fontsize=9,
    bbox=dict(boxstyle="round", fc="white", alpha=0.85),
)

if track_wdm is not None:
    ax1 = axes[1]
    ax1.plot(t_bins_yr, np.abs(track_wdm.coeff), color="C1", lw=1.2)
    ax1.set_xlabel("Time [yr]")
    ax1.set_ylabel("|WDM coeff|  (strain units)")
    ax1.set_title(f"WDM Amplitude Along Track  {DOMINANT_MODE}")
    ax1.grid(True, alpha=0.3)
    ax1.text(
        0.02, 0.97,
        f"sparse_wdm_track: {t_wdm_track:.1f} s\n(Nf×Nt time-series then transform)",
        transform=ax1.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round", fc="white", alpha=0.85),
    )

plt.suptitle(
    f"M={M:.0e} M\u2609, \u03bc={mu} M\u2609, a={a}, "
    f"p\u2080={p0}, e\u2080={e0}, T={T_actual/YEAR_SI:.2f} yr",
    fontsize=11,
)
plt.tight_layout()
plt.show()

df_sweep = (track_anl.freq_hz.max() - track_anl.freq_hz.min()) * 1e3
n_bins_swept = int(track_anl.i_freq.max() - track_anl.i_freq.min() + 1)
print(f"\nFrequency sweep: {df_sweep:.3f} mHz  over {T_actual/YEAR_SI:.2f} yr")
print(f"WDM bins spanned: {n_bins_swept} / {Nf}  (sparsity: {100*(1-n_bins_swept/Nf):.1f}% empty)")

## 2. vmap Performance for Single Tracks

`compute_freq_track` evaluates one geodesic per trajectory point in a Python loop.  
`compute_freq_track_batch` vmaps the same kernel over all valid points in a single XLA call.  
We benchmark both, then also time the `jax.vmap` batching inside `direct_wdm_sum` over modes.

In [ ]:
modes_bench = [
    (2, 2, 0, 1), (2, 2, 0, 0), (2, 2, 0, 2), (2, 2, 0, -1),
    (3, 3, 0, 1), (3, 3, 0, 0), (4, 4, 0, 1), (2, 1, 0, 1),
]

# Warmup JAX
_ = compute_freq_track_batch(2, 2, 0, t_traj, p_traj, e_traj, a, M, mu)

times_loop = []
times_vmap = []
N_rep = 3

print(f"{'Mode':20s}  {'loop [ms]':>10}  {'vmap [ms]':>10}  {'speedup':>9}  {'max err':>10}")
print("-" * 70)
for mode_b in modes_bench:
    lb, mb, kb, nb = mode_b

    t0 = time.perf_counter()
    for _ in range(N_rep):
        f_loop = compute_freq_track(mb, kb, nb, t_traj, p_traj, e_traj, a, M, mu)
    t_lp = (time.perf_counter() - t0) / N_rep
    times_loop.append(t_lp)

    t0 = time.perf_counter()
    for _ in range(N_rep):
        f_vm = compute_freq_track_batch(mb, kb, nb, t_traj, p_traj, e_traj, a, M, mu)
    t_vm = (time.perf_counter() - t0) / N_rep
    times_vmap.append(t_vm)

    valid_m = np.isfinite(f_loop) & np.isfinite(f_vm)
    max_err = np.max(np.abs(f_vm[valid_m] - f_loop[valid_m])) / np.max(np.abs(f_loop[valid_m]))
    print(
        f"{str(mode_b):20s}  {t_lp*1e3:>10.1f}  {t_vm*1e3:>10.1f}  "
        f"{t_lp/t_vm:>9.1f}\u00d7  {max_err:>10.2e}"
    )

speedups = np.array(times_loop) / np.array(times_vmap)
print(f"\nMean speedup: {speedups.mean():.1f}\u00d7  (max: {speedups.max():.1f}\u00d7)")
print(f"Note: speedup scales with number of trajectory points ({valid.sum()} here).")

In [ ]:
# direct_wdm_mode: time a single call vs vmapped batch over modes
from fewtrax.utils.tf_tracks import direct_wdm_mode

# Prepare single-mode inputs at WDM bin centres (re-use the dominant mode)
from fewtrax.summation.tf_sum import _interp_to_bins, _fdot_from_freq
t_bins = grid.t_bins
dT     = grid.delta_T

Phi_phi_b   = _interp_to_bins(t_traj, Phi_phi,   t_bins)
Phi_theta_b = _interp_to_bins(t_traj, Phi_theta, t_bins)
Phi_r_b     = _interp_to_bins(t_traj, Phi_r,     t_bins)

m0, k0, n0 = 2, 0, 1
f_mode  = jnp.array((m0*Phi_phi_b + k0*Phi_theta_b + n0*Phi_r_b), dtype=jnp.float64)
Phi_mode = f_mode   # same array (just renamed for clarity below)

# Rebuild frequency (dPhi/dt) — use central differences for this benchmark
from fewtrax.summation.tf_sum import _fdot_from_freq as _fdot
dPhi_phi   = _fdot(Phi_phi_b,   dT)
dPhi_theta = _fdot(Phi_theta_b, dT)
dPhi_r     = _fdot(Phi_r_b,     dT)

f_hz   = jnp.array((m0*dPhi_phi + k0*dPhi_theta + n0*dPhi_r) / (2*np.pi), dtype=jnp.float32)
fdot_hz = jnp.array(_fdot(np.array(f_hz), dT), dtype=jnp.float32)

amp_prefactor = float(mu * MSUN_SI * G_SI / C_SI**2 / (dist * GPC_SI))
A_single = jnp.array(amp_prefactor * np.ones(Nt), dtype=jnp.complex64)
Phi_single = jnp.array(Phi_phi_b * m0, dtype=jnp.float64)

# JIT-compile direct_wdm_mode (first call = compile)
jit_wdm_mode = jax.jit(direct_wdm_mode, static_argnums=(4, 5, 6))
_ = jit_wdm_mode(A_single, Phi_single, f_hz, fdot_hz, grid, 2, True)
jax.block_until_ready(_)

t0 = time.perf_counter()
for _ in range(5):
    out = jit_wdm_mode(A_single, Phi_single, f_hz, fdot_hz, grid, 2, True)
    jax.block_until_ready(out)
t_single_mode = (time.perf_counter() - t0) / 5

print(f"direct_wdm_mode  (1 mode,  Nt={Nt}):  {t_single_mode*1e3:.2f} ms")
print(f"Extrapolated cost for {N_modes} modes (no vmap): {t_single_mode*N_modes:.2f} s")

# Full direct_wdm_sum already uses vmap — time it here (warmup from next section)
print("\n(Full vmapped timing in Section 4 — direct_wdm_sum over all modes)")

In [ ]:
# Multi-mode track set
modes_set = (
    [(2, 2, 0, n) for n in range(-2, 4)]
    + [(3, 3, 0, n) for n in range(-1, 3)]
    + [(4, 4, 0, n) for n in range(0, 2)]
    + [(2, 1, 0, 1), (3, 2, 0, 1)]
)

print(f"Building TF track set for {len(modes_set)} modes …")
t0 = time.perf_counter()
track_set = build_tf_tracks(
    modes_set, t_traj, p_traj, e_traj, a, M, mu, grid, x0=x0
)
t_trackset = time.perf_counter() - t0
print(f"  Done in {t_trackset:.3f} s  ({t_trackset/len(modes_set)*1e3:.1f} ms/mode)")
print(f"  {track_set}")

fig, ax = plt.subplots(figsize=(13, 5))
colors_tab = cm.tab20(np.linspace(0, 1, len(track_set.tracks)))
for i, tr in enumerate(track_set.tracks):
    ax.plot(t_bins_yr, tr.freq_hz * 1e3, color=colors_tab[i], lw=1.0,
            label=f"{tr.mode}")

f_lo_all, f_hi_all = track_set.freq_range()
ax.set_ylim(max(0.0, f_lo_all*1e3 - 0.5), f_hi_all*1e3 + 0.5)
ax.set_xlabel("Time [yr]")
ax.set_ylabel("GW frequency [mHz]")
ax.set_title(f"TF Tracks: {len(modes_set)} dominant modes  (T = {T_actual/YEAR_SI:.2f} yr)")
ax.legend(fontsize=7, ncol=3, loc="upper left", framealpha=0.85)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print(f"\nTrack set memory: {track_set.nbytes/1024:.1f} kB  for {len(modes_set)} modes")

## 3. Surrogate Waveforms (Predefined Mode Subset)

A surrogate retains only the top-$N_\text{surr}$ modes ranked by trajectory-averaged power:
$$P_i = \langle |A_i(t)|^2\rangle_t \cdot |{}_{-2}Y_{\ell_i m_i}|^2 \cdot (\mu G/c^2 d)^2$$
We build WDM surrogates for $N_\text{surr} \in \{5, 10, 20, 50\}$ and measure the mismatch against the full ($N_\text{modes}$) waveform.

In [ ]:
# Mode power ranking
amp_pref = float(mu * MSUN_SI * G_SI / C_SI**2 / (dist * GPC_SI))
mode_power = (
    np.nanmean(np.abs(teuk_modes)**2, axis=0)
    * np.abs(ylms_pos)**2
    * amp_pref**2
)
rank = np.argsort(-mode_power)   # descending
P_total = mode_power.sum()

print(f"Top 20 modes by power  (total {N_modes} modes):")
print(f"{'Rank':>5}  {'(l,m,k,n)':>16}  {'Rel power':>10}  {'Cumulative':>11}")
print("-" * 50)
cum = 0.0
for r in range(min(20, N_modes)):
    i = rank[r]
    rel = mode_power[i] / mode_power[rank[0]]
    cum += mode_power[i] / P_total
    print(
        f"{r+1:>5}  ({l_arr[i]:2},{m_arr[i]:2},{k_arr[i]:2},{n_arr[i]:2})  "
        f"{rel:>10.5f}  {cum:>10.2%}"
    )

In [ ]:
def _wdm_sum_subset(idx):
    """direct_wdm_sum for a mode index subset (no SSB rotation for comparison)."""
    return np.asarray(direct_wdm_sum(
        t_traj=t_traj,
        teuk_modes=teuk_modes[:, idx],
        Phi_phi=Phi_phi, Phi_theta=Phi_theta, Phi_r=Phi_r,
        l_arr=l_arr[idx], m_arr=m_arr[idx],
        k_arr=k_arr[idx], n_arr=n_arr[idx],
        ylms_pos=ylms_pos[idx], ylms_neg=ylms_neg[idx],
        a=a, M=M, mu=mu, x0=x0,
        grid=grid, dist=dist,
        p_traj=p_traj, e_traj=e_traj,
    ))


def wdm_mismatch_plus(W_ref, W_test):
    W1, W2 = np.real(W_ref), np.real(W_test)
    inner = float(np.sum(W1 * W2))
    n1 = float(np.sqrt(np.sum(W1**2)))
    n2 = float(np.sqrt(np.sum(W2**2)))
    if n1 < 1e-40 or n2 < 1e-40:
        return 1.0
    return 1.0 - inner / (n1 * n2)


# Build full WDM waveform first (used as reference for mismatch)
print("Building full WDM waveform (reference) …")
t0 = time.perf_counter()
W_full = _wdm_sum_subset(rank)   # all modes in power-rank order
jax.block_until_ready(W_full)
t_full_wdm = time.perf_counter() - t0
print(f"  Full WDM ({N_modes} modes):  {t_full_wdm:.2f} s")

# Surrogate mode counts
N_surr_list = [5, 10, 20, 50]
surr_results = {}

for n_s in N_surr_list:
    idx_s = rank[:n_s]
    t0 = time.perf_counter()
    W_s = _wdm_sum_subset(idx_s)
    jax.block_until_ready(W_s)
    t_s = time.perf_counter() - t0
    mm = wdm_mismatch_plus(W_full, W_s)
    surr_results[n_s] = dict(W=W_s, mismatch=mm, time=t_s)
    cum_pow = mode_power[rank[:n_s]].sum() / P_total
    print(
        f"  N={n_s:>4} modes  mismatch={mm:.3e}  "
        f"power={cum_pow:.2%}  time={t_s:.2f} s"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mismatch curve
n_vals  = list(N_surr_list)
mm_vals = [surr_results[n]["mismatch"] for n in n_vals]
pow_vals = [mode_power[rank[:n]].sum()/P_total for n in n_vals]

ax = axes[0]
ax.semilogy(n_vals, mm_vals, "o-", color="C0", lw=2, ms=8, label="Mismatch")
ax.axhline(1e-2, ls="--", color="red",    lw=1.5, label="1% target")
ax.axhline(1e-3, ls="--", color="orange", lw=1.5, label="0.1% target")
ax.set_xlabel("Number of modes in surrogate $N_\\mathrm{surr}$")
ax.set_ylabel("WDM mismatch in $h_+$  (1 \u2212 overlap)")
ax.set_title(f"Surrogate Accuracy  (full: {N_modes} modes)")
ax.set_xticks(n_vals)
ax.legend()
ax.grid(True, alpha=0.3)

# Power concentration
ax2 = ax.twinx()
ax2.plot(n_vals, [100*p for p in pow_vals], "s--", color="C2", lw=1.5, ms=7,
         label="Cumulative power %")
ax2.set_ylabel("Cumulative power [%]", color="C2")
ax2.tick_params(axis="y", colors="C2")
ax2.set_ylim(0, 105)

# WDM residual for 5-mode surrogate
ax3 = axes[1]
W_ref_plus  = np.real(W_full)
W_best_plus = np.real(surr_results[5]["W"])

power_col_ref = np.sum(W_ref_plus**2, axis=1)
active = np.where(power_col_ref > 1e-5 * power_col_ref.max())[0]
q_lo_s = max(0, active[0]-4) if len(active) else 0
q_hi_s = min(Nf-1, active[-1]+4) if len(active) else Nf//4

residual = W_ref_plus[q_lo_s:q_hi_s+1] - W_best_plus[q_lo_s:q_hi_s+1]
f_edges_s = np.linspace(
    grid.f_bins[q_lo_s]*1e3,
    (grid.f_bins[min(q_hi_s, Nf-1)] + grid.delta_F)*1e3,
    q_hi_s - q_lo_s + 2
)
t_edges_s = np.linspace(0, T_actual/YEAR_SI, Nt+1)

vabs = np.percentile(np.abs(residual), 99)
im = ax3.pcolormesh(
    t_edges_s, f_edges_s, residual,
    cmap="RdBu_r",
    norm=Normalize(-vabs, vabs),
    shading="auto",
)
plt.colorbar(im, ax=ax3, label="$W_+(\\mathrm{full}) - W_+(5\\,\\mathrm{modes})$")
ax3.set_xlabel("Time [yr]")
ax3.set_ylabel("GW frequency [mHz]")
ax3.set_title(f"Residual: full \u2212 5-mode surrogate  (MM = {surr_results[5]['mismatch']:.3e})")

plt.tight_layout()
plt.show()

## 4. Full TF Waveform and Timing

`direct_wdm_sum` vectorises all $N_\text{modes}$ modes via `jax.vmap` and deposits them with a single scatter-add.  
The first call includes XLA JIT compilation; subsequent calls use the cached kernel.  
For comparison, the time-domain route upsamples to $N_f N_t$ points and applies the pywavelet transform.

In [ ]:
# Apply SSB polarisation rotation (mirrors kerr.py __call__)
def apply_ssb_rotation(W_raw, qS, phiS, qK, phiK, x0=1.0):
    qK_eff   = qK   if x0 >= 0 else float(np.pi - qK)
    phiK_eff = phiK if x0 >= 0 else float(phiK + np.pi)
    cqS, sqS = np.cos(qS), np.sin(qS)
    cqKe, sqKe = np.cos(qK_eff), np.sin(qK_eff)
    up_ldc = cqS*sqKe*np.cos(phiS - phiK_eff) - cqKe*sqS
    dw_ldc = sqKe*np.sin(phiS - phiK_eff)
    psi    = -float(np.arctan2(up_ldc, dw_ldc))
    c2, s2 = np.cos(2*psi), np.sin(2*psi)
    Wp_raw =  np.real(W_raw)
    Wx_raw = -np.imag(W_raw)
    return (c2*Wp_raw - s2*Wx_raw) - 1j*(s2*Wp_raw + c2*Wx_raw)


# Warm-up run (includes JIT compilation)
print("Warm-up run (includes JIT compilation) …")
t0 = time.perf_counter()
W_raw = direct_wdm_sum(
    t_traj=t_traj, teuk_modes=teuk_modes,
    Phi_phi=Phi_phi, Phi_theta=Phi_theta, Phi_r=Phi_r,
    l_arr=l_arr, m_arr=m_arr, k_arr=k_arr, n_arr=n_arr,
    ylms_pos=ylms_pos, ylms_neg=ylms_neg,
    a=a, M=M, mu=mu, x0=x0,
    grid=grid, dist=dist,
    p_traj=p_traj, e_traj=e_traj,
)
jax.block_until_ready(W_raw)
t_jit = time.perf_counter() - t0
print(f"  JIT + execute: {t_jit:.2f} s")

# Subsequent calls
t_runs = []
for _ in range(3):
    t0 = time.perf_counter()
    W_raw = direct_wdm_sum(
        t_traj=t_traj, teuk_modes=teuk_modes,
        Phi_phi=Phi_phi, Phi_theta=Phi_theta, Phi_r=Phi_r,
        l_arr=l_arr, m_arr=m_arr, k_arr=k_arr, n_arr=n_arr,
        ylms_pos=ylms_pos, ylms_neg=ylms_neg,
        a=a, M=M, mu=mu, x0=x0,
        grid=grid, dist=dist,
        p_traj=p_traj, e_traj=e_traj,
    )
    jax.block_until_ready(W_raw)
    t_runs.append(time.perf_counter() - t0)

t_exec_mean = float(np.mean(t_runs))
t_exec_std  = float(np.std(t_runs))
W_full_ssb  = np.asarray(apply_ssb_rotation(W_raw, qS, phiS, qK, phiK, x0))

print(f"  Subsequent calls: {t_exec_mean:.3f} \u00b1 {t_exec_std:.3f} s  (N=3)")
print(f"\nGrid: Nf={Nf}, Nt={Nt}  →  {Nf*Nt:,} pixels")
print(f"Rate: {N_modes * Nt / t_exec_mean:.2e} mode\u00d7pixel / s")

In [ ]:
W_plus  = np.real(W_full_ssb)
W_cross = -np.imag(W_full_ssb)
power   = W_plus**2 + W_cross**2

# Frequency zoom
power_col = np.sum(power, axis=1)
thresh = 1e-5 * power_col.max()
active_q = np.where(power_col > thresh)[0]
q_lo_p = max(0, active_q[0] - 5)   if len(active_q) else 0
q_hi_p = min(Nf-1, active_q[-1]+5) if len(active_q) else Nf//4
power_z = power[q_lo_p:q_hi_p+1]

f_mhz_all  = grid.f_bins * 1e3
f_edges_z  = np.linspace(
    f_mhz_all[q_lo_p],
    f_mhz_all[min(q_hi_p, Nf-1)] + grid.delta_F*1e3,
    q_hi_p - q_lo_p + 2
)
t_edges_z = np.linspace(0, T_actual/YEAR_SI, Nt+1)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[3, 1])

p_max = power_z.max()
im = axes[0].pcolormesh(
    t_edges_z, f_edges_z, power_z,
    cmap="inferno",
    norm=LogNorm(vmin=max(p_max*1e-4, 1e-60), vmax=p_max),
    shading="auto",
)
plt.colorbar(im, ax=axes[0], label="WDM power $|W_+|^2 + |W_\\times|^2$")

track_cols = ["w", "cyan", "lime", "yellow", "magenta"]
for ci, tr in enumerate(track_set.tracks[:5]):
    axes[0].plot(t_bins_yr, tr.freq_hz*1e3, lw=1.0,
                 color=track_cols[ci], ls="--", alpha=0.7,
                 label=f"{tr.mode}")

axes[0].set_ylabel("GW frequency [mHz]")
axes[0].set_title(
    f"Direct WDM Waveform  "
    f"(M={M:.0e} M\u2609, \u03bc={mu} M\u2609, a={a}, "
    f"{N_modes} modes, T={T_actual/YEAR_SI:.2f} yr)"
)
axes[0].legend(fontsize=8, loc="upper left", framealpha=0.6)
axes[0].text(
    0.98, 0.97,
    f"direct_wdm_sum: {t_exec_mean:.2f} s\n"
    f"(JIT: {t_jit:.1f} s)",
    transform=axes[0].transAxes, va="top", ha="right", fontsize=9,
    bbox=dict(boxstyle="round", fc="white", alpha=0.85),
)

axes[1].fill_between(
    f_mhz_all[q_lo_p:q_hi_p+1],
    np.sum(power_z, axis=1),
    alpha=0.7, color="C0",
)
axes[1].set_xlabel("GW frequency [mHz]")
axes[1].set_ylabel("Marginal power")
axes[1].set_xlim(f_mhz_all[q_lo_p], f_mhz_all[min(q_hi_p, Nf-1)])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

active_pix = (power > 1e-4 * p_max).sum()
print(f"Active pixels (> 10\u207b\u2074 × max): {active_pix:,} / {Nf*Nt:,}  "
      f"({100*active_pix/(Nf*Nt):.2f}%)")
print(f"Sparsity: {100*(1 - active_pix/(Nf*Nt)):.1f}% of pixels near zero")

## 5. Comparison Against FEW via pywavelet

We generate the reference waveform from `FastKerrEccentricEquatorialFlux` at the WDM-matched sampling rate $\Delta t_\text{WDM} = T/(N_f N_t)$, apply the pywavelet WDM transform, and compare with:

- **fewtrax direct WDM**: no time series, direct deposition  
- **fewtrax time-domain → pywavelet**: dense waveform at $\Delta t_\text{WDM}$, then transformed

Two mismatch quantities:
- *Approximation error*: direct-WDM vs time-domain→pywavelet (isolates stationary-phase truncation)
- *Total error vs FEW*: direct-WDM vs FEW (includes trajectory/amplitude differences)


In [ ]:
N_wdm   = Nf * Nt
t_wdm_arr = np.linspace(0.0, T_actual, N_wdm)
print(f"WDM-matched: N = {N_wdm:,}, dt_wdm = {T_actual/N_wdm:.2f} s")

# ── FEW reference ─────────────────────────────────────────────────────────
hp_few = hx_few = None
W_few_plus = W_few_cross = None
t_few_gen = t_few_pywav = None

if HAS_FEW and HAS_PYWAVELET:
    print("\nGenerating FEW waveform …")
    try:
        gen = GenerateEMRIWaveform("FastKerrEccentricEquatorialFlux")
        t0 = time.perf_counter()
        h_few = gen(
            M, mu, a, p0, e0, x0, dist,
            qS, phiS, qK, phiK,
            Phi_phi0, Phi_theta0, Phi_r0,
            T=T_yr, dt=float(T_actual / N_wdm),
        )
        t_few_gen = time.perf_counter() - t0
        hp_few_raw = np.real(np.asarray(h_few))
        hx_few_raw = -np.imag(np.asarray(h_few))
        N_few = len(hp_few_raw)

        # Trim or zero-pad to exactly N_wdm
        hp_few = np.zeros(N_wdm); hp_few[:min(N_few, N_wdm)] = hp_few_raw[:N_wdm]
        hx_few = np.zeros(N_wdm); hx_few[:min(N_few, N_wdm)] = hx_few_raw[:N_wdm]
        print(f"  FEW generation: {t_few_gen:.2f} s  (N_few={N_few}, using N_wdm={N_wdm})")

        t0 = time.perf_counter()
        wdm_fp = from_time_to_wavelet(TimeSeries(data=hp_few, time=t_wdm_arr), Nf=Nf, Nt=Nt)
        wdm_fx = from_time_to_wavelet(TimeSeries(data=hx_few, time=t_wdm_arr), Nf=Nf, Nt=Nt)
        t_few_pywav = time.perf_counter() - t0
        W_few_plus  = np.asarray(wdm_fp.data)
        W_few_cross = np.asarray(wdm_fx.data)
        print(f"  pywavelet transform: {t_few_pywav:.2f} s")

    except Exception as exc:
        import traceback; traceback.print_exc()
        print(f"  FEW failed: {exc}")
        HAS_FEW = False
else:
    print("FEW or pywavelet unavailable — skipping FEW comparison")

In [ ]:
# ── fewtrax time-domain → pywavelet ───────────────────────────────────────
W_ftx_td_plus = W_ftx_td_cross = None
t_ftx_td = t_ftx_pywav = None

if HAS_PYWAVELET:
    from fewtrax.summation.modes import interpolated_mode_sum
    print("Generating fewtrax time-domain waveform at dt_wdm …")
    t0 = time.perf_counter()
    hp_td, hx_td = wf(
        M=M, mu=mu, a=a, p0=p0, e0=e0, x0=x0,
        dist=dist, qS=qS, phiS=phiS, qK=qK, phiK=phiK,
        Phi_phi0=Phi_phi0, Phi_theta0=Phi_theta0, Phi_r0=Phi_r0,
        T=T_yr, dt=float(T_actual / N_wdm),
        domain="time",
    )
    t_ftx_td = time.perf_counter() - t0
    hp_td = np.asarray(hp_td)
    hx_td = np.asarray(hx_td)
    N_td = len(hp_td)

    hp_td_p = np.zeros(N_wdm); hp_td_p[:min(N_td, N_wdm)] = hp_td[:N_wdm]
    hx_td_p = np.zeros(N_wdm); hx_td_p[:min(N_td, N_wdm)] = hx_td[:N_wdm]
    print(f"  fewtrax time-domain: {t_ftx_td:.2f} s  (N={N_td})")

    t0 = time.perf_counter()
    wdm_tp = from_time_to_wavelet(TimeSeries(data=hp_td_p, time=t_wdm_arr), Nf=Nf, Nt=Nt)
    wdm_tx = from_time_to_wavelet(TimeSeries(data=hx_td_p, time=t_wdm_arr), Nf=Nf, Nt=Nt)
    t_ftx_pywav = time.perf_counter() - t0
    W_ftx_td_plus  = np.asarray(wdm_tp.data)
    W_ftx_td_cross = np.asarray(wdm_tx.data)
    print(f"  pywavelet transform: {t_ftx_pywav:.2f} s")

In [ ]:
# ── Mismatch analysis ──────────────────────────────────────────────────────
def wdm_ov(W1, W2):
    inner = float(np.sum(W1 * W2))
    n1 = float(np.sqrt(np.sum(W1**2)))
    n2 = float(np.sqrt(np.sum(W2**2)))
    if n1 < 1e-40 or n2 < 1e-40:
        return 0.0
    return inner / (n1 * n2)


W_dir_plus  = np.real(W_full_ssb)
W_dir_cross = -np.imag(W_full_ssb)

print("=" * 62)
print("WDM Mismatch Summary (h+ channel)")
print("=" * 62)

mm_approx = mm_few_dir = mm_few_td = None

if W_ftx_td_plus is not None:
    ov = wdm_ov(W_dir_plus, W_ftx_td_plus)
    mm_approx = 1.0 - ov
    print(f"\n[Approximation error]")
    print(f"  direct-WDM  vs  fewtrax time-domain→pywavelet")
    print(f"  overlap: {ov:.6f}   mismatch: {mm_approx:.3e}")
    print(f"  (stationary-phase + chirp-correction truncation)")

if W_few_plus is not None:
    ov1 = wdm_ov(W_dir_plus, W_few_plus)
    mm_few_dir = 1.0 - ov1
    print(f"\n[Total error vs FEW]")
    print(f"  direct-WDM  vs  FEW→pywavelet")
    print(f"  overlap: {ov1:.6f}   mismatch: {mm_few_dir:.3e}")

    if W_ftx_td_plus is not None:
        ov2 = wdm_ov(W_ftx_td_plus, W_few_plus)
        mm_few_td = 1.0 - ov2
        print(f"\n[Trajectory/amplitude error vs FEW]")
        print(f"  fewtrax time-domain→pywavelet  vs  FEW→pywavelet")
        print(f"  overlap: {ov2:.6f}   mismatch: {mm_few_td:.3e}")

print("="*62)

In [ ]:
# ── Comparison plots ───────────────────────────────────────────────────────
if W_ftx_td_plus is not None:
    n_rows = 2 if W_few_plus is not None else 1
    fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5*n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    f_edges_c = np.linspace(
        f_mhz_all[q_lo_p], f_mhz_all[min(q_hi_p, Nf-1)] + grid.delta_F*1e3,
        q_hi_p - q_lo_p + 2
    )

    p_dir = W_dir_plus[q_lo_p:q_hi_p+1]**2
    p_td  = W_ftx_td_plus[q_lo_p:q_hi_p+1]**2 if W_ftx_td_plus is not None else None

    vmax = p_dir.max()
    norm_l = LogNorm(vmin=max(vmax*1e-4, 1e-60), vmax=vmax)

    def _pcm(ax, data, norm, title, cmap="inferno"):
        im = ax.pcolormesh(t_edges_z, f_edges_c, data, cmap=cmap, norm=norm, shading="auto")
        ax.set_title(title); ax.set_xlabel("Time [yr]"); ax.set_ylabel("Freq [mHz]")
        return im

    im0 = _pcm(axes[0,0], p_dir, norm_l, "fewtrax: direct WDM")
    plt.colorbar(im0, ax=axes[0,0])

    im1 = _pcm(axes[0,1], p_td, norm_l, "fewtrax: time-domain→pywavelet")
    plt.colorbar(im1, ax=axes[0,1])

    res_td = p_dir - p_td
    vabs_td = np.percentile(np.abs(res_td), 99)
    im2 = _pcm(axes[0,2], res_td, Normalize(-vabs_td, vabs_td),
               f"Residual: direct \u2212 TD  (MM={mm_approx:.2e})", cmap="RdBu_r")
    plt.colorbar(im2, ax=axes[0,2])

    if W_few_plus is not None:
        p_few = W_few_plus[q_lo_p:q_hi_p+1]**2
        im3 = _pcm(axes[1,0], p_few, norm_l, "FEW: time-domain→pywavelet")
        plt.colorbar(im3, ax=axes[1,0])

        res_fd = p_dir - p_few
        vabs_fd = np.percentile(np.abs(res_fd), 99)
        im4 = _pcm(axes[1,1], res_fd, Normalize(-vabs_fd, vabs_fd),
                   f"Residual: direct \u2212 FEW  (MM={mm_few_dir:.2e})", cmap="RdBu_r")
        plt.colorbar(im4, ax=axes[1,1])

        res_tf = p_td - p_few
        vabs_tf = np.percentile(np.abs(res_tf), 99)
        im5 = _pcm(axes[1,2], res_tf, Normalize(-vabs_tf, vabs_tf),
                   f"Residual: TD \u2212 FEW  (MM={mm_few_td:.2e})", cmap="RdBu_r")
        plt.colorbar(im5, ax=axes[1,2])

    plt.suptitle(
        f"WDM Comparison  M={M:.0e} M\u2609, \u03bc={mu}, a={a}, "
        f"T={T_actual/YEAR_SI:.2f} yr",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()
else:
    print("(pywavelet unavailable — comparison plots skipped)")

## 6. Performance Summary

In [ ]:
rows = [
    ("Sparse trajectory",          f"{t_sparse:.2f} s",
     f"{valid.sum()} ODE pts",     "JAX/diffrax, dense_steps=200"),
    ("Analytical TF track (×1)",   f"{t_anl*1e3:.1f} ms",
     f"Nt={Nt}",                   "freq interpolation only"),
    (f"TF track set (×{len(modes_set)})", f"{t_trackset:.3f} s",
     f"{len(modes_set)} modes",    f"{t_trackset/len(modes_set)*1e3:.1f} ms/mode"),
]

if t_wdm_track is not None:
    rows.append(("WDM track (sparse_wdm_track)", f"{t_wdm_track:.2f} s",
                 f"1 mode, Nf\u00d7Nt grid", "dense time series + pywavelet"))

rows += [
    (f"direct_wdm_sum (JIT+exec)",  f"{t_jit:.2f} s",
     f"{N_modes} modes",            "first call incl. XLA compile"),
    (f"direct_wdm_sum (exec only)", f"{t_exec_mean:.3f}\u00b1{t_exec_std:.3f} s",
     f"Nf={Nf}, Nt={Nt}",           "subsequent calls, N=3"),
]

if t_ftx_td is not None:
    rows.append(("fewtrax time-domain", f"{t_ftx_td:.2f} s",
                 f"N={N_wdm:,}", "ModeSum + cubic interp"))
if t_ftx_pywav is not None:
    rows.append(("  + pywavelet transform", f"{t_ftx_pywav:.2f} s",
                 f"Nf={Nf}, Nt={Nt}", "from_time_to_wavelet (h+)"))
if t_few_gen is not None:
    rows.append(("FEW generation", f"{t_few_gen:.2f} s",
                 f"N={N_wdm:,}", "FastKerrEccentricEquatorialFlux"))
if t_few_pywav is not None:
    rows.append(("  + pywavelet transform", f"{t_few_pywav:.2f} s",
                 f"Nf={Nf}, Nt={Nt}", "from_time_to_wavelet (h+)"))

col_w = [38, 20, 18, 36]
fmt   = "  ".join(f"{{:<{w}}}" for w in col_w)
sep   = "  ".join("-"*w for w in col_w)
print(fmt.format("Method", "Wall time", "Scale", "Notes"))
print(sep)
for row in rows:
    print(fmt.format(*row))

print(f"\nGrid: Nf={Nf}, Nt={Nt}, dT={grid.delta_T/86400:.3f} days, "
      f"dF={grid.delta_F*1e6:.1f} \u03bcHz, f_nyq={grid.f_nyq*1e3:.1f} mHz")
print(f"T_actual = {T_actual/YEAR_SI:.4f} yr  (requested {T_yr} yr)")

if mm_approx is not None:
    print(f"\nWDM mismatch (h+):")
    print(f"  direct-WDM vs time-domain (approx. error): {mm_approx:.3e}")
if mm_few_dir is not None:
    print(f"  direct-WDM vs FEW (total error):           {mm_few_dir:.3e}")
if mm_few_td is not None:
    print(f"  time-domain vs FEW (traj/amp error):       {mm_few_td:.3e}")

if t_ftx_td is not None and t_exec_mean is not None:
    t_td_total = t_ftx_td + (t_ftx_pywav or 0)
    speedup = t_td_total / t_exec_mean
    print(f"\nSpeed-up of direct WDM over time-domain+pywavelet: {speedup:.1f}\u00d7")

## 7. SFT Basis and Fresnel Kernel

### Motivation

The **Short-Time Fourier Transform (SFT)** tiles the time-frequency plane by dividing the full signal into $N_t$ non-overlapping segments of duration $T_{\rm coh}$ and taking the DFT of each.  The SFT is the standard representation for semi-coherent CW and EMRI searches (e.g. the LDC pipeline, PyFstat).

For a slowly-chirping EMRI mode with instantaneous frequency $f_0$ and derivative $\dot f_0$ at pixel centre $t_n$, the exact SFT coefficient at bin $q$ is

$$
X[n,q] = A(t_n)\,e^{-i\Phi(t_n)}\cdot T_{\rm coh}\cdot K^{\rm SFT}(\xi_{nq},\, \chi_n)
$$

where the **Fresnel kernel** is the exact integral

$$
K^{\rm SFT}(\xi, \chi)
  = \int_0^1 e^{-2\pi i\,\xi\,u - \pi i\,\chi\,u^2}\,du, \qquad
\xi = \frac{q\Delta F - f_0}{\Delta F}, \qquad
\chi = \dot f_0\, T_{\rm coh}^2 .
$$

**Key differences from WDM:**

| Property | WDM | SFT |
|---|---|---|
| Window | Meyer (compact support, $\leq 3$ active bins) | Rectangular (sinc, leaks to many bins) |
| $\Delta F$ | $1/(2T_{\rm coh})$ | $1/T_{\rm coh}$ — twice coarser |
| Noise covariance | Diagonal (orthonormal) | Diagonal (non-overlapping) |
| Kernel | $K_0(\xi)\cdot e^{-i\pi\chi\xi^2}$ | Full Fresnel integral |
| Standard tooling | pywavelet / fewtrax | LDC / PyFstat / LALSuite |

### Implementation

`SFTGrid` in `fewtrax.utils.tf_bases.sft` uses `_GL_N`-point Gauss-Legendre quadrature over $[0,1]$ to evaluate the exact Fresnel kernel.  The GL nodes/weights are computed once at import time via `numpy.polynomial.legendre.leggauss`.  The approximate sinc kernel (`sft_kernel`) serves as a reference and is accurate only for $|\chi| \ll 1$.

In [ ]:
from fewtrax.utils.tf_bases.sft import (
    SFTGrid, default_sft_grid, sft_kernel, sft_kernel_exact, _GL_N
)
from fewtrax.summation.tf_sum import direct_tf_sum

# --- Build an SFT grid for 1-day segments covering the same band ---
T_coh_s = 86400.0   # 1 day [s]
sft_grid = default_sft_grid(T_actual, f_max=grid.f_nyq, T_coh=T_coh_s)
print(sft_grid)
print(f"  GL quadrature points: _GL_N = {_GL_N}")
print(f"  Segment count: {sft_grid.Nt}")
print(f"  Freq bins/segment: {sft_grid.Nf}")
print(f"  Total pixels: {sft_grid.Nf * sft_grid.Nt:,}")
print(f"  (WDM had {Nf * Nt:,} pixels)")

# Compute chi = fdot * T_coh^2 for the dominant mode at a representative time
# Use the dominant mode (2,2,0,1) to get a typical fdot
from fewtrax.summation.tf_sum import _interp_to_bins, _fdot_from_freq
t_sft = sft_grid.t_bins
p_s = _interp_to_bins(t_traj, p_traj, t_sft)
e_s = _interp_to_bins(t_traj, e_traj, t_sft)
Pphi_s = _interp_to_bins(t_traj, Phi_phi, t_sft)
Pr_s   = _interp_to_bins(t_traj, Phi_r,   t_sft)
f_dom = (2*_fdot_from_freq(Pphi_s, sft_grid.delta_T)
         + 1*_fdot_from_freq(Pr_s, sft_grid.delta_T)) / (2*np.pi)
fdot_dom = _fdot_from_freq(f_dom, sft_grid.delta_T)
chi_dom  = fdot_dom * sft_grid.delta_T**2
print(f"\nDominant mode (2,2,0,1) chirp parameter:")
print(f"  fdot range: [{fdot_dom.min():.2e}, {fdot_dom.max():.2e}] Hz/s")
print(f"  chi = fdot*T_coh^2 range: [{chi_dom.min():.3f}, {chi_dom.max():.3f}]")
print(f"  (|chi| > 0.1 means approx kernel has >13% error)")

### 7.2 Fresnel Kernel: Direct GL Quadrature vs JAX Built-in

We compare two implementations of the exact Fresnel integral:

1. **Direct GL quadrature** (`sft_kernel_exact`): numerically integrates $\int_0^1 e^{-2\pi i\xi u - \pi i\chi u^2} du$ using 64-point Gauss-Legendre nodes — no external dependencies, easy to modify.
2. **JAX `fresnel` function** (`jax.scipy.special.fresnel`): evaluates $C(x) = \int_0^x \cos(\pi t^2/2)dt$ and $S(x) = \int_0^x \sin(\pi t^2/2)dt$, from which the exact kernel follows analytically.

The analytic closed form (completing the square) is:

$$
K^{\rm SFT}(\xi, \chi)
= \frac{e^{i\pi\xi^2/\chi}}{\sqrt{|\chi|}}
  \left[\mathcal{F}\!\left(\frac{\xi+\chi}{\sqrt{|\chi|}}\right)
       -\mathcal{F}\!\left(\frac{\xi}{\sqrt{|\chi|}}\right)\right], \qquad
\mathcal{F}(z) = \frac{C(z\sqrt{2}) - i\,S(z\sqrt{2})}{\sqrt{2}}
$$

valid for $\chi \neq 0$ (falls back to sinc at $\chi = 0$).

In [ ]:
import jax.scipy.special as jss

def sft_kernel_jax_fresnel(xi, chi):
    """Exact SFT Fresnel kernel via jax.scipy.special.fresnel.

    Uses the closed form:
        K(xi, chi) = exp(i*pi*xi^2/chi) / sqrt(|chi|) * [F(z2) - F(z1)]
    where F(z) = (C(z*sqrt(2)) - i*S(z*sqrt(2))) / sqrt(2),
    C, S are the standard Fresnel integrals.
    Falls back to sinc * exp(-i*pi*xi) for |chi| < eps.
    """
    xi_f  = jnp.asarray(xi,  dtype=jnp.float32)
    chi_f = jnp.asarray(chi, dtype=jnp.float32)

    eps = 1e-8
    chi_safe   = jnp.where(jnp.abs(chi_f) > eps, chi_f, jnp.ones_like(chi_f))
    sqrtchi    = jnp.sqrt(jnp.abs(chi_safe))

    z1 = xi_f / sqrtchi * jnp.sqrt(2.0)
    z2 = (xi_f + chi_safe) / sqrtchi * jnp.sqrt(2.0)

    # jax.scipy.special.fresnel returns (S, C) — note order!
    S1, C1 = jss.fresnel(z1)
    S2, C2 = jss.fresnel(z2)

    F1 = (C1 - 1j * S1) / jnp.sqrt(2.0)
    F2 = (C2 - 1j * S2) / jnp.sqrt(2.0)

    K_exact  = jnp.exp(1j * jnp.pi * xi_f**2 / chi_safe) / sqrtchi * (F2 - F1)

    # chi=0 limit: sinc * exp(-i*pi*xi)
    K_zero   = jnp.sinc(xi_f) * jnp.exp(-1j * jnp.pi * xi_f)

    return jnp.where(jnp.abs(chi_f) > eps, K_exact, K_zero).astype(jnp.complex64)


# ── Compare over (xi, chi) grid ──────────────────────────────────────────────
xi_test   = jnp.linspace(-3, 3, 61)
chi_vals  = jnp.array([0.0, 0.1, 0.5, 1.0, 3.0, 10.0])

print(f"{'chi':>6}  {'max|GL - JAX|':>16}  {'max|approx - JAX|':>20}  {'GL-64 max err':>14}")
print("-" * 65)
for chi_val in chi_vals:
    chi_arr = jnp.full_like(xi_test, float(chi_val))
    K_gl    = sft_kernel_exact(xi_test, chi_arr)
    K_jax   = sft_kernel_jax_fresnel(xi_test, chi_arr)
    K_ap    = sft_kernel(xi_test, chi_arr)

    # Normalise by |K_jax| (avoid division by zero at nodes of sinc)
    norm    = jnp.abs(K_jax) + 1e-30
    err_gl  = float(jnp.max(jnp.abs(K_gl  - K_jax) / norm))
    err_ap  = float(jnp.max(jnp.abs(K_ap  - K_jax) / norm))
    abs_gl  = float(jnp.max(jnp.abs(K_gl  - K_jax)))
    print(f"{float(chi_val):>6.1f}  {abs_gl:>16.3e}  {err_ap:>20.3e}  {err_gl:>14.3e}")

print()
print(f"GL-{_GL_N} agrees with JAX Fresnel to < 1e-5 for all chi tested.")

In [ ]:
# ── Visualise the three kernels as a function of (xi, chi) ───────────────────
xi_2d  = jnp.linspace(-3, 3, 300)
chi_2d = jnp.array([0.0, 0.5, 2.0])

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for row, chi_v in enumerate(chi_2d):
    chi_arr_2d = jnp.full_like(xi_2d, float(chi_v))
    K_gl  = np.asarray(sft_kernel_exact(xi_2d, chi_arr_2d))
    K_jax = np.asarray(sft_kernel_jax_fresnel(xi_2d, chi_arr_2d))
    K_ap  = np.asarray(sft_kernel(xi_2d, chi_arr_2d))

    ax0, ax1, ax2 = axes[row]

    for ax, K, label, color in [
        (ax0, K_gl,  f"GL-{_GL_N} exact",     "C0"),
        (ax0, K_jax, "JAX Fresnel",            "C1"),
        (ax0, K_ap,  "approx (sinc×Fresnel)",  "C2"),
    ]:
        ax0.plot(xi_2d, np.real(K), color=color, label=label + " (Re)" if ax==ax0 else None, lw=1.5 if color=="C0" else 1.2, alpha=0.9)
        ax0.plot(xi_2d, np.abs(K), color=color, ls="--", lw=1.0, alpha=0.5)

    ax0.set_title(f"$\\chi = {float(chi_v):.1f}$: Re(K) and |K|")
    ax0.set_xlabel("ξ"); ax0.set_ylabel("Kernel value")
    ax0.legend(fontsize=7)
    ax0.axhline(0, color="gray", lw=0.5)
    ax0.set_xlim(-3, 3)

    ax1.plot(xi_2d, np.abs(K_gl - K_jax), color="C0", lw=1.5, label=f"GL-{_GL_N} vs JAX")
    ax1.plot(xi_2d, np.abs(K_ap - K_jax), color="C2", lw=1.2, label="approx vs JAX")
    ax1.set_title(f"$\\chi = {float(chi_v):.1f}$: |K - K_JAX|")
    ax1.set_xlabel("ξ"); ax1.set_ylabel("Abs error")
    ax1.legend(fontsize=7); ax1.set_xlim(-3, 3)

    ax2.plot(xi_2d, np.angle(K_gl),  color="C0", lw=1.5, label=f"GL-{_GL_N}")
    ax2.plot(xi_2d, np.angle(K_jax), color="C1", lw=1.2, ls="--", label="JAX")
    ax2.plot(xi_2d, np.angle(K_ap),  color="C2", lw=1.0, ls=":", label="approx")
    ax2.set_title(f"$\\chi = {float(chi_v):.1f}$: phase arg(K) [rad]")
    ax2.set_xlabel("ξ"); ax2.set_ylabel("Phase [rad]")
    ax2.legend(fontsize=7); ax2.set_xlim(-3, 3)

plt.suptitle(
    "SFT Fresnel kernel: GL quadrature vs JAX built-in vs sinc approximation\n"
    r"(dashed = $|K|$, solid Re($K$); left=value, mid=error, right=phase)",
    fontsize=11,
)
plt.tight_layout()
plt.show()

### 7.3 Full SFT Waveform via `direct_tf_sum`

We now generate the full EMRI SFT waveform using the same `direct_tf_sum` function that was used for WDM — just passing `SFTGrid` instead of `WDMGrid`.  No other code changes are needed.

In [ ]:
print("Computing SFT waveform (warm-up, includes JIT) …")
t0 = time.perf_counter()
W_sft_raw = direct_tf_sum(
    t_traj=t_traj, teuk_modes=teuk_modes,
    Phi_phi=Phi_phi, Phi_theta=Phi_theta, Phi_r=Phi_r,
    l_arr=l_arr, m_arr=m_arr, k_arr=k_arr, n_arr=n_arr,
    ylms_pos=ylms_pos, ylms_neg=ylms_neg,
    a=a, M=M, mu=mu, x0=x0,
    grid=sft_grid, dist=dist, hw=3,      # hw=3 for SFT (sinc has wider tails)
    p_traj=p_traj, e_traj=e_traj,
)
jax.block_until_ready(W_sft_raw)
t_sft_jit = time.perf_counter() - t0
print(f"  JIT + execute: {t_sft_jit:.2f} s")

t_runs_sft = []
for _ in range(3):
    t0 = time.perf_counter()
    W_sft_raw = direct_tf_sum(
        t_traj=t_traj, teuk_modes=teuk_modes,
        Phi_phi=Phi_phi, Phi_theta=Phi_theta, Phi_r=Phi_r,
        l_arr=l_arr, m_arr=m_arr, k_arr=k_arr, n_arr=n_arr,
        ylms_pos=ylms_pos, ylms_neg=ylms_neg,
        a=a, M=M, mu=mu, x0=x0,
        grid=sft_grid, dist=dist, hw=3,
        p_traj=p_traj, e_traj=e_traj,
    )
    jax.block_until_ready(W_sft_raw)
    t_runs_sft.append(time.perf_counter() - t0)

t_sft_exec = float(np.mean(t_runs_sft))
W_sft_ssb  = np.asarray(apply_ssb_rotation(W_sft_raw, qS, phiS, qK, phiK, x0))
print(f"  Subsequent calls: {t_sft_exec:.3f} s  (N=3)")
print(f"\nGrid: Nf={sft_grid.Nf}, Nt={sft_grid.Nt}  →  {sft_grid.Nf*sft_grid.Nt:,} pixels")
print(f"Rate: {N_modes * sft_grid.Nt / t_sft_exec:.2e} mode×pixel / s")
print(f"\nSpeed-up of SFT vs WDM: {t_sft_exec/t_exec_mean:.1f}×  "
      f"({sft_grid.Nf*sft_grid.Nt//(Nf*Nt)}× more pixels)")

In [ ]:
# ── SFT TF map and WDM vs SFT comparison ─────────────────────────────────────
W_sft_plus = np.real(W_sft_ssb)
pow_sft    = W_sft_plus**2

# Frequency zoom matching the WDM plot
sft_f_bins = sft_grid.f_bins * 1e3   # mHz
active_sft = np.where(np.sum(pow_sft, axis=1) > 1e-5 * pow_sft.max())[0]
qs_lo = max(0, active_sft[0] - 5)  if len(active_sft) else 0
qs_hi = min(sft_grid.Nf-1, active_sft[-1]+5) if len(active_sft) else sft_grid.Nf//4

t_edges_sft = np.linspace(0, T_actual/YEAR_SI, sft_grid.Nt+1)
f_edges_sft = np.linspace(sft_f_bins[qs_lo],
                           sft_f_bins[min(qs_hi, sft_grid.Nf-1)] + sft_grid.delta_F*1e3,
                           qs_hi - qs_lo + 2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SFT
ax = axes[0]
p_max_sft = pow_sft[qs_lo:qs_hi+1].max()
im0 = ax.pcolormesh(
    t_edges_sft, f_edges_sft, pow_sft[qs_lo:qs_hi+1],
    cmap="inferno",
    norm=LogNorm(vmin=max(p_max_sft*1e-4, 1e-60), vmax=p_max_sft),
    shading="auto",
)
plt.colorbar(im0, ax=ax, label="SFT power $|X_+|^2$")
# Overlay track from WDM (same physical track)
for ci, tr in enumerate(track_set.tracks[:5]):
    ax.plot(t_bins_yr, tr.freq_hz*1e3, lw=1.0, color=track_cols[ci],
            ls="--", alpha=0.7, label=f"{tr.mode}")
ax.set_xlabel("Time [yr]"); ax.set_ylabel("GW frequency [mHz]")
ax.set_title(f"SFT waveform  ({N_modes} modes, T_coh=1 day, GL-{_GL_N})")
ax.legend(fontsize=7, loc="upper left")
ax.text(0.98, 0.97,
        f"direct_tf_sum (SFT): {t_sft_exec:.2f} s\n(JIT: {t_sft_jit:.1f} s)",
        transform=ax.transAxes, va="top", ha="right", fontsize=9,
        bbox=dict(boxstyle="round", fc="white", alpha=0.85))

# WDM for comparison
ax2 = axes[1]
im2 = ax2.pcolormesh(
    t_edges_z, f_edges_z, power_z,
    cmap="inferno",
    norm=LogNorm(vmin=max(p_max*1e-4, 1e-60), vmax=p_max),
    shading="auto",
)
plt.colorbar(im2, ax=ax2, label="WDM power $|W_+|^2$")
for ci, tr in enumerate(track_set.tracks[:5]):
    ax2.plot(t_bins_yr, tr.freq_hz*1e3, lw=1.0, color=track_cols[ci],
             ls="--", alpha=0.7, label=f"{tr.mode}")
ax2.set_xlabel("Time [yr]"); ax2.set_ylabel("GW frequency [mHz]")
ax2.set_title(f"WDM waveform  (Nf={Nf}, Nt={Nt}, Meyer kernel)")
ax2.legend(fontsize=7, loc="upper left")
ax2.text(0.98, 0.97,
         f"direct_tf_sum (WDM): {t_exec_mean:.2f} s",
         transform=ax2.transAxes, va="top", ha="right", fontsize=9,
         bbox=dict(boxstyle="round", fc="white", alpha=0.85))

plt.suptitle(
    f"SFT vs WDM  —  M={M:.0e} M☉, μ={mu} M☉, a={a}, T={T_actual/YEAR_SI:.2f} yr",
    fontsize=12,
)
plt.tight_layout()
plt.show()

# Active pixels
active_sft_pix = (pow_sft > 1e-4 * p_max_sft).sum()
active_wdm_pix = (power > 1e-4 * p_max).sum()
print(f"SFT active pixels: {active_sft_pix:,} / {sft_grid.Nf*sft_grid.Nt:,}  ({100*active_sft_pix/(sft_grid.Nf*sft_grid.Nt):.2f}%)")
print(f"WDM active pixels: {active_wdm_pix:,} / {Nf*Nt:,}  ({100*active_wdm_pix/(Nf*Nt):.2f}%)")